In [2]:
!pip install librosa numpy pandas torch parselmouth


Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [1 lines of output]
      error in googleads setup command: use_2to3 is invalid.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [7]:
import os
import librosa
import numpy as np
import parselmouth  # for jitter/shimmer

SR = 16000  # sample rate

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=SR, mono=True)
    
    # -------------------- MFCC --------------------
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfccs_mean = np.mean(mfccs.T, axis=0)  # (40,)
    
    # -------------------- Prosody --------------------
    # Pitch
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]  # remove unvoiced
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    
    # Jitter & Shimmer
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    # Energy
    energy = np.mean(y**2)
    
    # Speaking rate (syllables/sec approximation using onset detection)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0
    
    # Combine all features
    features = np.hstack([mfccs_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    return features


In [8]:
import pandas as pd
import numpy as np

root_folder = r'User_Voice'

X = []
y_user = []
y_question = []

for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_path):
        continue
    for file in sorted(os.listdir(user_path)):
        if file.endswith('.wav'):
            file_path = os.path.join(user_path, file)
            features = extract_features(file_path)
            X.append(features)
            
            y_user.append(user_folder)
            q_num = int(file.split('.')[0])  # 01.wav -> 1
            y_question.append(q_num)

X = np.array(X)
y_user = np.array(y_user)
y_question = np.array(y_question)

print("Features extracted!")
print("X shape:", X.shape)


Features extracted!
X shape: (230, 46)


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class VoiceDataset(Dataset):
    def __init__(self, X, y_user, y_question, user_dict, q_dict):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_user = torch.tensor([user_dict[u] for u in y_user], dtype=torch.long)
        self.y_question = torch.tensor([q_dict[q] for q in y_question], dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y_user[idx], self.y_question[idx]

# Mapping labels
user_dict = {u:i for i,u in enumerate(sorted(set(y_user)))}
q_dict = {q:i for i,q in enumerate(sorted(set(y_question)))}

dataset = VoiceDataset(X, y_user, y_question, user_dict, q_dict)

# Split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [10]:
import torch.nn as nn

class VoiceMLP(nn.Module):
    def __init__(self, input_dim, num_users, num_questions):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        self.user_head = nn.Linear(128, num_users)
        self.q_head = nn.Linear(128, num_questions)
    
    def forward(self, x):
        x = self.shared(x)
        return self.user_head(x), self.q_head(x)

model = VoiceMLP(input_dim=X.shape[1], num_users=len(user_dict), num_questions=len(q_dict))


In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_user, batch_q in train_loader:
        batch_x, batch_user, batch_q = batch_x.to(device), batch_user.to(device), batch_q.to(device)
        optimizer.zero_grad()
        out_user, out_q = model(batch_x)
        loss_user = criterion(out_user, batch_user)
        loss_q = criterion(out_q, batch_q)
        loss = loss_user + loss_q
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1, Loss: 8.7667
Epoch 2, Loss: 5.3274
Epoch 3, Loss: 4.6650
Epoch 4, Loss: 4.1958
Epoch 5, Loss: 3.7752
Epoch 6, Loss: 3.6872
Epoch 7, Loss: 3.2882
Epoch 8, Loss: 3.0915
Epoch 9, Loss: 2.9064
Epoch 10, Loss: 2.7494
Epoch 11, Loss: 2.6879
Epoch 12, Loss: 2.5644
Epoch 13, Loss: 2.5249
Epoch 14, Loss: 2.3331
Epoch 15, Loss: 2.2813
Epoch 16, Loss: 2.1247
Epoch 17, Loss: 2.1208
Epoch 18, Loss: 1.9718
Epoch 19, Loss: 1.9516
Epoch 20, Loss: 1.9548
Epoch 21, Loss: 1.8473
Epoch 22, Loss: 1.7762
Epoch 23, Loss: 1.7064
Epoch 24, Loss: 1.6522
Epoch 25, Loss: 1.6269
Epoch 26, Loss: 1.5499
Epoch 27, Loss: 1.7055
Epoch 28, Loss: 1.5480
Epoch 29, Loss: 1.5071
Epoch 30, Loss: 1.4646


In [12]:
from sklearn.metrics import accuracy_score

model.eval()
all_user_preds, all_user_labels = [], []
all_q_preds, all_q_labels = [], []

with torch.no_grad():
    for batch_x, batch_user, batch_q in test_loader:
        batch_x = batch_x.to(device)
        out_user, out_q = model(batch_x)
        user_pred = out_user.argmax(dim=1).cpu().numpy()
        q_pred = out_q.argmax(dim=1).cpu().numpy()
        all_user_preds.extend(user_pred)
        all_user_labels.extend(batch_user.numpy())
        all_q_preds.extend(q_pred)
        all_q_labels.extend(batch_q.numpy())

print("Speaker recognition accuracy:", accuracy_score(all_user_labels, all_user_preds))
print("Question classification accuracy:", accuracy_score(all_q_labels, all_q_preds))


Speaker recognition accuracy: 0.8695652173913043
Question classification accuracy: 0.10869565217391304
